# Stage 6a — Interactive exploration

Walks through the model end-to-end with concrete visualisations:
the true pendulum, the heatmap encoding, the encoder's posterior,
the latent rollout, the decoded q(t), and the pendulum as a swinging
stick.

Run cells top-to-bottom. Anywhere you see a `# tweak` marker you
can change values and re-run downstream cells to play around.

Run from this directory (`stages/06a_heatmap_render/`).
Loads the most recent `logs/*/*/model.pt` by default.

In [1]:
# Force matplotlib's inline backend. Safety net for environments where
# some earlier import (or VSCode default) left the backend as 'Agg',
# which would silently make plt.show() a no-op.
%matplotlib inline

## 1. Setup — load the trained model

In [2]:
import sys
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchdiffeq import odeint

# Make `train` importable so we can reuse its classes/helpers.
sys.path.insert(0, str(Path.cwd()))
from train import (PendulumTruth, VariationalEncoderGRU, ODEFunc,
                   decode_q, render_heatmap, true_pendulum_field)

torch.set_grad_enabled(False)

# Find the most recent run with a saved checkpoint.
candidates = sorted(Path('logs').glob('*/*/model.pt'))
assert candidates, 'No model.pt found -- run `uv run python train.py` first.'
ckpt_path = candidates[-1]  # tweak: pick a different index for a different run
print(f'loading checkpoint: {ckpt_path}')
ckpt = torch.load(ckpt_path, weights_only=False)
cfg = ckpt['config']

loading checkpoint: logs/2026-06-03/11-50-21/model.pt


In [3]:
# Unpack config and build the helpers the model needs.
horizon = cfg['data']['window_horizon']
n_obs = cfg['data']['n_obs']
heatmap_n = cfg['data']['heatmap_n']
heatmap_range = cfg['data']['heatmap_range']
heatmap_sigma = cfg['data']['heatmap_sigma']
obs_noise = cfg['data']['obs_noise']
latent_dim = cfg['model']['latent_dim']
hidden = cfg['model']['hidden']

positions = torch.linspace(-heatmap_range, heatmap_range, heatmap_n)
amps_all = sorted(set(cfg['data']['amplitudes']) | set(cfg['data']['test_amplitudes']))
truth = PendulumTruth(amps_all, cfg['data']['t_max'], cfg['data']['n_lut'],
                      cfg['train']['solver'])

encoder = VariationalEncoderGRU(obs_dim=heatmap_n, hidden_size=hidden,
                                latent_dim=latent_dim)
encoder.load_state_dict(ckpt['encoder_state']); encoder.eval()
func = ODEFunc(dim=latent_dim, hidden=hidden)
func.load_state_dict(ckpt['func_state']); func.eval()
print('model loaded.  train amps:', cfg['data']['amplitudes'],
      ' test amps:', cfg['data']['test_amplitudes'])

model loaded.  train amps: [0.3, 1.0, 1.5]  test amps: [0.3, 0.6, 1.0, 1.5]


## 2. The pendulum as ground truth

Pick an amplitude and look at the trajectory we're trying to learn.
Two views:

- `q(t)` over the 6-second observation window.
- The `(q, p)` phase portrait — a closed ellipse, parameterized by amplitude.

In [ ]:
# tweak: change the amplitude here to look at a different pendulum.
A_show = 1.0

# Extended time range: show ~3 encoder windows worth of truth so multiple
# full periods are visible. (Truth LUT covers [0, t_max=20 s].)
t_extended = min(horizon * 3.0, cfg['data']['t_max'] - 0.5)
t_grid = torch.linspace(0.0, t_extended, 600)
qp = truth.state(torch.tensor([A_show]), t_grid)[:, 0]   # (T, 2)
q_true, p_true = qp[:, 0], qp[:, 1]

# Sample an example encoder window so we can SEE where the model's
# observations land on the trajectory. Same seed as Section 4 so the
# points match what we look at in detail there.
torch.manual_seed(42)
obs_times_show = (torch.rand(n_obs) * horizon).sort().values
q_at_obs_show = truth.state(torch.tensor([A_show]),
                            obs_times_show)[..., 0].squeeze(-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
ax = axes[0]
ax.plot(t_grid, q_true, lw=2, label='true q(t)')
ax.axvspan(0, horizon, color='tab:green', alpha=0.10,
           label=f'encoder obs window [0, {horizon}] s')
ax.scatter(obs_times_show, q_at_obs_show, color='red', s=40, zorder=3,
           label=f'{n_obs} encoder obs')
ax.set_xlabel('t'); ax.set_ylabel('q')
ax.set_title(f'q(t)  at A={A_show}  (extended view, {t_extended:.0f} s)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(q_true, p_true, lw=2)
ax.set_xlabel('q'); ax.set_ylabel('p'); ax.set_aspect('equal')
ax.set_title(f'(q, p) phase portrait at A={A_show}')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 3. From angle to observation: the heatmap encoding

The model never sees `q` directly. It sees a length-64 array `o[i]`
with a Gaussian bump centred at `q`. Two views:

- One snapshot: heatmap at a chosen `t`, with the true `q` marked.
- A vertical "movie strip": time runs down, position runs across,
  the bump traces out `q(t)`.

In [ ]:
# tweak: pick a single time within the window
t_show = 2.0

q_at_t = truth.state(torch.tensor([A_show]), torch.tensor([t_show]))[0, 0, 0].item()
heatmap = render_heatmap(torch.tensor(q_at_t), positions, heatmap_sigma).numpy()

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(positions.numpy(), heatmap, lw=2)
ax.axvline(q_at_t, color='red', lw=1, ls='--', label=f'true q = {q_at_t:+.2f}')
ax.set_xlabel('pixel position (q-units)')
ax.set_ylabel('o[i]')
ax.set_title(f'Heatmap observation at A={A_show}, t={t_show}')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Movie strip: dense grid of heatmaps stacked vertically.
t_dense = torch.linspace(0.0, horizon, 120)
q_dense = truth.state(torch.tensor([A_show]), t_dense)[:, 0, 0]   # (T,)
movie = render_heatmap(q_dense, positions, heatmap_sigma).numpy()   # (T, N_pix)

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(movie, aspect='auto', origin='upper',
          extent=[-heatmap_range, heatmap_range, horizon, 0.0],
          cmap='viridis', interpolation='nearest')
ax.plot(q_dense.numpy(), t_dense.numpy(), color='red', lw=1, label='true q(t)')
ax.set_xlabel('pixel position (q-units)')
ax.set_ylabel('t')
ax.set_title(f'Heatmap movie of q(t) at A={A_show}  (clean, no noise)')
ax.legend()
fig.tight_layout()
plt.show()

## 4. One encoder window, fully unpacked

What the encoder actually sees per training sample: 10 sparse,
irregularly spaced heatmap observations. We:

1. Draw the random sample times on the true `q(t)` curve as red dots.
2. Show the 10 corresponding heatmap rows (noisy, as the encoder sees them).
3. Run the encoder and print its posterior `(μ, σ)` over `z₀`.

In [ ]:
# tweak: re-run this cell to draw a fresh random observation window.
torch.manual_seed(42)  # set to None for fresh randomness each time

obs_local = (torch.rand(1, n_obs) * horizon).sort(dim=1).values   # (1, N)
q_obs_clean = truth.state(torch.tensor([A_show]), obs_local.T)[..., 0].T   # (1, N)
o_obs_clean = render_heatmap(q_obs_clean, positions, heatmap_sigma)         # (1, N, N_pix)
o_obs = o_obs_clean + obs_noise * torch.randn_like(o_obs_clean)            # add noise

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t_grid, q_true, color='lightgray', lw=2, label='true q(t)')
ax.scatter(obs_local[0].numpy(), q_obs_clean[0].numpy(),
           color='red', s=40, zorder=3, label=f'{n_obs} obs times')
ax.set_xlabel('t'); ax.set_ylabel('q')
ax.set_title(f'Where the {n_obs} obs land on the true trajectory  (A={A_show})')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Show the 10 noisy heatmap rows the encoder actually consumes.
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(o_obs[0].numpy(), aspect='auto', origin='lower',
          extent=[-heatmap_range, heatmap_range, 0, n_obs],
          cmap='viridis', interpolation='nearest')
ax.scatter(q_obs_clean[0].numpy(),
           torch.arange(n_obs).numpy() + 0.5,
           color='red', s=30, label='true q at each obs')
ax.set_xlabel('pixel position (q-units)')
ax.set_ylabel('obs index  (0 = earliest)')
ax.set_title(f'Encoder input -- {n_obs} noisy heatmaps  (A={A_show})')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Run the encoder. It returns (mu, logvar), each (1, latent_dim=2).
mu, logvar = encoder(obs_local, o_obs)
std = (0.5 * logvar).exp()
print('Posterior parameters at t=0 for this encoder window:')
print(f'  mu     = {mu[0].numpy().round(3)}')
print(f'  sigma  = {std[0].numpy().round(3)}')
print()
# Draw 5 samples from N(mu, sigma^2) to see how spread out z0 is.
for k in range(5):
    z0_sample = mu + std * torch.randn_like(mu)
    print(f'  sample {k}: z0 = {z0_sample[0].numpy().round(3)}'
          f'    (z[0] interpreted as predicted q at t=0)')
print()
print(f'True q at t=0: {truth.state(torch.tensor([A_show]), torch.tensor([0.0]))[0, 0, 0].item():.3f}')

## 5. From z₀ to a full predicted trajectory

Take one `z₀` (the posterior mean `μ`) and roll it forward through the
ODE. Three views:

1. The latent trajectory in `(z[0], z[1])` space.
2. The decoded `q_pred(t) = z[0](t)` against true `q(t)`.
3. The predicted heatmap movie, side by side with the true heatmap movie.

In [ ]:
# Use the posterior mean as the point estimate.
z0 = mu                                                           # (1, latent_dim)
z_traj = odeint(func, z0, t_grid, method=cfg['train']['solver'])  # (T, 1, latent_dim)
z_traj = z_traj.squeeze(1)                                        # (T, latent_dim)
q_pred = decode_q(z_traj)                                         # (T,)

# Plot 1: latent phase portrait
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(q_true, p_true, color='lightgray', lw=3, label='true (q, p)')
ax.plot(z_traj[:, 0], z_traj[:, 1], '--', lw=1.6, label='latent (z[0], z[1])')
ax.set_xlabel('z[0] / q'); ax.set_ylabel('z[1] / p')
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Latent orbit vs true (q, p)')
ax.grid(alpha=0.3)

# Plot 2: q vs t (over the extended view)
ax = axes[1]
ax.plot(t_grid, q_true, color='lightgray', lw=3, label='true q(t)')
ax.plot(t_grid, q_pred, '--', lw=1.6, label='predicted q(t)')
ax.axvspan(0, horizon, color='tab:green', alpha=0.08,
           label=f'training horizon ({horizon}s)')
ax.set_xlabel('t'); ax.set_ylabel('q')
ax.set_title('Predicted vs true angle  (beyond shaded region = extrapolation)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Heatmap movie: predicted vs true, side by side, over the extended view.
movie_pred = render_heatmap(q_pred, positions, heatmap_sigma).numpy()
movie_true = render_heatmap(q_true, positions, heatmap_sigma).numpy()
T_end = float(t_grid[-1].item())

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, m, title in zip(axes, [movie_true, movie_pred],
                        ['true heatmaps', 'predicted heatmaps']):
    ax.imshow(m, aspect='auto', origin='upper',
              extent=[-heatmap_range, heatmap_range, T_end, 0.0],
              cmap='viridis', interpolation='nearest', vmin=0, vmax=1)
    # Mark the encoder's training horizon: everything below the line is
    # extrapolation beyond what the model saw at training time.
    ax.axhline(horizon, color='white', lw=1, ls='--', alpha=0.7)
    ax.set_xlabel('pixel position (q-units)')
    ax.set_title(title)
axes[0].set_ylabel('t')
fig.suptitle('Rendered q_pred(t) vs true q(t)  '
             f'(dashed line at t={horizon}s = training horizon; below = extrapolation)')
fig.tight_layout()
plt.show()

## 6. The pendulum as a picture

Translate the angle into actual pendulum poses. For a handful of
snapshots across the 6-second window, draw two pendulums side by side
(true vs predicted) on a 2D canvas.

Convention: `q = 0` means hanging straight down. Positive `q` swings to
the right.

In [ ]:
def draw_pendulum(ax, q, *, color='black', alpha=1.0, L=1.0):
    """Draw a pendulum rod from (0,0) hanging at angle q from vertical.
    q=0 -> straight down.  Positive q swings right."""
    x = L * np.sin(q)
    y = -L * np.cos(q)
    ax.plot([0, x], [0, y], color=color, lw=2.5, alpha=alpha)
    ax.scatter([x], [y], s=140, color=color, zorder=3, alpha=alpha,
               edgecolor='black', linewidths=0.5)
    ax.scatter([0], [0], s=25, color='dimgray', zorder=4)

def setup_pendulum_axes(ax, L=1.0):
    ax.set_xlim(-1.4*L, 1.4*L)
    ax.set_ylim(-1.3*L, 0.3*L)
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', lw=0.5, alpha=0.3)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.3)
    ax.set_xticks([]); ax.set_yticks([])

# tweak: choose a few timestamps to compare poses at.
snapshot_ts = np.linspace(0.0, horizon, 8)

# Get true & predicted q at each snapshot time.
snap_t = torch.tensor(snapshot_ts, dtype=torch.float32)
q_true_snap = truth.state(torch.tensor([A_show]), snap_t)[:, 0, 0].numpy()
z_snap = odeint(func, z0, snap_t, method=cfg['train']['solver']).squeeze(1)
q_pred_snap = decode_q(z_snap).numpy()

fig, axes = plt.subplots(2, len(snapshot_ts), figsize=(2.3 * len(snapshot_ts), 5))
for k, (t, q_t, q_p) in enumerate(zip(snapshot_ts, q_true_snap, q_pred_snap)):
    setup_pendulum_axes(axes[0, k]); setup_pendulum_axes(axes[1, k])
    draw_pendulum(axes[0, k], q_t, color='dimgray')
    draw_pendulum(axes[1, k], q_p, color='tab:blue')
    axes[0, k].set_title(f't={t:.2f}\nq_true={q_t:+.2f}', fontsize=9)
    axes[1, k].set_title(f'q_pred={q_p:+.2f}', fontsize=9)
axes[0, 0].set_ylabel('true', fontsize=11)
axes[1, 0].set_ylabel('predicted', fontsize=11)
fig.suptitle(f'Pendulum poses across the window  (A={A_show})')
fig.tight_layout()
plt.show()

## 7. Uncertainty in pendulum space

Instead of a single point prediction, draw 32 posterior samples and
show them as **overlaid faint pendulums** at a chosen timestamp. The
wider the fan, the more uncertain the model is at that moment.

In [ ]:
# tweak: which moment(s) to visualize, and how many samples.
fan_ts = [0.5, 2.0, 4.0, 5.5]
n_samples = 32

fan_t_tensor = torch.tensor(fan_ts, dtype=torch.float32)

# Reparametrized samples: same encoder posterior, many z0 draws.
z0_samples = mu + std * torch.randn(n_samples, latent_dim)
# Roll out all samples in one batched odeint call.
z_samples = odeint(func, z0_samples, fan_t_tensor,
                   method=cfg['train']['solver'])           # (T_fan, S, latent)
q_samples = decode_q(z_samples).numpy()                    # (T_fan, S)
q_true_fan = truth.state(torch.tensor([A_show]), fan_t_tensor)[:, 0, 0].numpy()

fig, axes = plt.subplots(1, len(fan_ts), figsize=(2.6 * len(fan_ts), 3.4))
for k, (ax, t, q_t) in enumerate(zip(axes, fan_ts, q_true_fan)):
    setup_pendulum_axes(ax)
    # Faint pendulums for each sample.
    for s in range(n_samples):
        draw_pendulum(ax, q_samples[k, s], color='tab:blue', alpha=0.12)
    # True pendulum on top.
    draw_pendulum(ax, q_t, color='red', alpha=0.95)
    ax.set_title(f't={t:.2f}\n true q={q_t:+.2f}', fontsize=9)
fig.suptitle(f'Posterior fan of pendulum poses  (blue = samples, red = truth, A={A_show})')
fig.tight_layout()
plt.show()

## Things to play with

- **Cell 2.tweak**: switch `A_show` to 0.3, 0.6, or 1.5 to see different orbits.
- **Cell 3.tweak**: pick different `t_show` values.
- **Cell 4.tweak**: change the seed (or set to `None`) for fresh random obs draws.
- **Cell 6.tweak**: change `snapshot_ts` to zoom in on a specific time range.
- **Cell 7.tweak**: more samples for a fatter posterior fan; different `fan_ts`.